# Milestone 3 lab, notebook 5 of 8: The update that came late

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| **5. The update that came late** (you are here) | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

The merge fixed every repeat. One problem from milestone 2 is of a different kind: a change that the load **never reads at all**. To see it clearly, this notebook first runs the pipeline with the lookback switched off.

In [1]:
import pandas as pd

from lab_helpers import api, arden_time, bars, lab_warehouse, money, run_pipeline, show_doc, table, true_day_totals

from daily_orders.report import daily_report, day_totals

## Step 1: Three mornings, no lookback

In [2]:
con = lab_warehouse(fresh=True)
run_pipeline(con, "2026-06-08", "2026-06-09", "2026-06-10", lookback_days=0)

def wrong_days():
    report, truth = day_totals(daily_report(con)), true_day_totals()
    rows = []
    for day, right in truth.items():
        shown = report[day]
        if (shown["orders"], shown["revenue_cents"]) != (right["orders"], right["revenue_cents"]):
            rows.append({"day": day, "report": f"{shown['orders']} orders, {money(shown['revenue_cents'])}",
                         "truth": f"{right['orders']} orders, {money(right['revenue_cents'])}"})
    return pd.DataFrame(rows) if rows else "The report equals the truth on every day. ✅"

wrong_days()

,day,report,truth
0,2026-06-08,"25 orders, $591.75","24 orders, $573.49"


One day is wrong: Monday 8 June has one order too many. There are no duplicates any more, so this is something else. Find an order whose state in the warehouse differs from the source.

In [3]:
warehouse_orders = dict(con.execute("SELECT order_id, status FROM core.orders WHERE order_date = '2026-06-08'").fetchall())

for order_id, status in warehouse_orders.items():
    source = api.get(f"/orders/{order_id}").json()
    if source["status"] != status:
        print(f"Order {order_id}:  warehouse says '{status}',  the source says '{source['status']}'")
        late = source

Order 1244:  warehouse says 'out_for_delivery',  the source says 'cancelled'


## Step 2: Why was it never read?

Look at when the order changed, and at what each run asked for.

In [4]:
print(f"Order {late['order_id']} was cancelled at  {late['updated_at']}  ({arden_time(late['updated_at']):%A %H:%M} in Arden)\n")

table(con, "SELECT run_id, asked_since, watermark_after, rows_landed FROM ops.load_runs ORDER BY run_id")

Order 1244 was cancelled at  2026-06-09T02:35:00Z  (Monday 21:35 in Arden)



,run_id,asked_since,watermark_after,rows_landed
0,2026-06-08_1,NaT,2026-06-08 03:48:53+00:00,226
1,2026-06-09_1,2026-06-08 03:48:53+00:00,2026-06-09 04:17:31+00:00,29
2,2026-06-10_1,2026-06-09 04:17:31+00:00,2026-06-10 04:11:39+00:00,38


Read the last row. Wednesday's run asked for orders changed since `2026-06-09 04:17:31`. The cancellation has an `updated_at` of `2026-06-09 02:35:00`. That is **before** the watermark, so the question "what changed since my watermark?" does not include it.

But why did Tuesday's run not read it? It ran at 07:00 on Tuesday, hours after 02:35 UTC. Look at what Tuesday's run landed for this order.

In [5]:
table(con, """
    SELECT run_id, payload->>'status' AS status, payload->>'updated_at' AS updated_at
    FROM raw.orders_landing WHERE (payload->>'order_id')::int = %s ORDER BY landing_id
""", [late["order_id"]])

,run_id,status,updated_at
0,2026-06-09_1,out_for_delivery,2026-06-09T02:00:00Z


On Tuesday morning the order system still said `out_for_delivery`. The cancellation was not in the order system yet.

Here is the story. On Monday evening the customer refused the bag at the door. The driver recorded it at 21:35 in the app, but the phone was offline. The app sent the change the next morning at 10:15, and the order system stored it with the time it **happened**: Monday 21:35.

```
Mon 21:35   the order is cancelled at the door          <- updated_at says this
Tue 07:00   Tuesday's run. The API still shows 'out_for_delivery'. Watermark moves to Mon 23:17.
Tue 10:15   the driver's phone connects. The API now shows 'cancelled', updated_at Mon 21:35
Wed 07:00   Wednesday's run asks: changed since Mon 23:17?   -> Mon 21:35 is earlier. Not returned.
```

The change arrived **after the window that should have caught it**. Such a row is called a **late row**.

## Step 3: Look back

The fix is simple. Do not start exactly at the watermark. Start a little earlier.

```python
since = watermark - timedelta(days=2)
```

That is a **lookback**. Each run re-reads the last two days, so a change that arrives late is still inside the window.

Run the same three mornings again, with the lookback the pipeline uses by default.

In [6]:
con.close()
con = lab_warehouse(fresh=True)
run_pipeline(con, "2026-06-08", "2026-06-09", "2026-06-10")          # lookback_days = 2 is the default

wrong_days()

'The report equals the truth on every day. ✅'

In [7]:
table(con, "SELECT run_id, asked_since, watermark_after, rows_landed, orders_inserted, orders_updated FROM ops.load_runs ORDER BY run_id")

,run_id,asked_since,watermark_after,rows_landed,orders_inserted,orders_updated
0,2026-06-08_1,NaT,2026-06-08 03:48:53+00:00,226,226,0
1,2026-06-09_1,2026-06-06 03:48:53+00:00,2026-06-09 04:17:31+00:00,102,25,4
2,2026-06-10_1,2026-06-07 04:17:31+00:00,2026-06-10 04:11:39+00:00,93,31,8


Wednesday's run now asks from `2026-06-07`, two days before the watermark. Order 1244's cancellation is inside that window.

## Step 4: What the lookback costs, and why it is safe

A lookback re-reads rows the warehouse already holds. Compare `rows_landed` with the orders that really changed.

In [8]:
runs = table(con, "SELECT run_id, rows_landed, orders_inserted + orders_updated AS orders_changed FROM ops.load_runs WHERE run_id <> '2026-06-08_1' ORDER BY run_id")
runs["read_again_for_nothing"] = runs["rows_landed"] - runs["orders_changed"]
runs

,run_id,rows_landed,orders_changed,read_again_for_nothing
0,2026-06-09_1,102,29,73
1,2026-06-10_1,93,39,54


Most of what each run reads is already known. In milestone 2 that would have been a disaster: every one of those rows would have been appended and counted again.

Here it costs a few extra API calls and nothing else, because the merge leaves a known version alone.

> A lookback re-reads rows. It is safe only because the write is idempotent.

That is why this notebook comes after notebook 3, and not before.

In [9]:
bars({"Full load (milestone 1)": 251, "No lookback (milestone 2)": 29, "2-day lookback (now)": int(runs["rows_landed"][0])}, unit=" rows read on Tuesday")

Full load (milestone 1)    ████████████████████████████████████████ 251 rows read on Tuesday
No lookback (milestone 2)  █████ 29 rows read on Tuesday
2-day lookback (now)       ████████████████ 102 rows read on Tuesday


## Step 5: How many days?

Choose the lookback from real data, not from a feeling. How late was this change?

In [10]:
happened = arden_time(late["updated_at"])
print(f"It happened:           {happened:%A %H:%M}")
print("The API showed it at:  Tuesday 10:15   (see the business page, order 1244)")
print("Late by:               12 hours and 40 minutes")

It happened:           Monday 21:35
The API showed it at:  Tuesday 10:15   (see the business page, order 1244)
Late by:               12 hours and 40 minutes


One day of lookback would have caught it. The pipeline uses two, to leave room. That is a decision, and it has a limit: a change that arrives more than two days late is still missed.

In [11]:
show_doc("LLD.md", "5. The lookback")

**From `docs/LLD.md`**

> ## 5. The lookback
> 
> ```python
> since = watermark - timedelta(days=LOOKBACK_DAYS)     # LOOKBACK_DAYS = 2
> ```
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*
> 
> The cancellation has an `updated_at` of Monday 21:35. The API first showed it on Tuesday at 10:15, after Tuesday's run. By then the watermark was already past Monday 21:35.
> 
> | Run of 10 June | Asks for orders changed since | Reads order 1244's cancellation? |
> |---|---|---|
> | Without lookback | `2026-06-09T04:17:31Z` | No. For ever |
> | With a 2-day lookback | `2026-06-07T04:17:31Z` | Yes |

In [12]:
con.close()

## Two new words

| Word | Meaning | Here |
|---|---|---|
| **Late row** | "A late row reaches the source after the window that should have caught it." The general name is late-arriving data | Order 1244's cancellation: it happened Monday 21:35 and arrived Tuesday 10:15 |
| **Lookback** | The window starts N days before the watermark. Also called a look-back window | N = 2 days |

The definitions are from the D8Loop chapter [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills).

## What you now know

- A late row has an `updated_at` in the past. A plain watermark never finds it.
- A lookback re-reads a few days before the watermark, and catches it.
- The lookback reads many rows for nothing. That is only safe with an idempotent write.
- Choose the number of days from how late your data really arrives.

**Say it in an interview:** "A driver's phone was offline, so a cancellation reached the API twelve hours late with an old `updated_at`. My watermark had already passed it. I added a two-day lookback: each run re-reads two days before the watermark. That is only safe because my write is a MERGE, so re-read rows change nothing."

Next: [notebook 6](06_fix_one_day.ipynb) fixes one day of the report without touching the others.